In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()  


True

In [3]:
## load pdf and print 100 characters
# specify the path to the PDF file
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "telecom_guide.pdf"  

loader = PyPDFLoader(pdf_path)
pages = loader.load()
# print number of pages
print(f"loading pdf {pdf_path} with {len(pages)} pages")
print("\n --- first 100 characters of the first page ---")
print(pages[0].page_content[:100])


loading pdf telecom_guide.pdf with 9 pages

 --- first 100 characters of the first page ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Ca


In [6]:
## Chunk PDF Documents into smaller pieces for processing
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "] # tries paragraph boundaries -> line -> sentence -> word
)
chunks = text_splitter.split_documents(pages)
## print number of chunks
print(f"Split the document into {len(chunks)} chunks")
print("\nfirst chunk: \n")
print(chunks[0].page_content)


Split the document into 37 chunks

first chunk: 

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [7]:
## Create Embedings for the chunks of the PDF document and store them in chroma DB 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Created vector store with {len(chunks)} chunks")
# vector collection count 
print(f"Vector store ready and collection count :: {vector_store._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16832.12it/s]


Created vector store with 37 chunks
Vector store ready and collection count :: 37


In [8]:
## stored is done , now lets do Retrieval 
retriver = vector_store.as_retriever(search_kwargs={"k": 3}) ## returns top 3 most relevant chunks for a query
test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriver.invoke(test_query)
print(f"Retrieved chunks for the query '{test_query}':")
for i, chunk in enumerate(retrieved):
    print(f"\nChunk {i+1}:\n{chunk.page_content}")

Retrieved chunks for the query 'What is VoLTE and how does it improve call quality?':

Chunk 1:
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and

Chunk 2:
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data O

In [ ]:
## Build a rag pipeline 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnalvePassthrough
from langchain_groq import ChatGroq


# --- Heker : join retrived chunks into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])


SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the user's query based on the retrieved chunks. 
If the context is not sufficient, respond with 'I don't know.'

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"), # dynamically filled with the user's question
])
## Create LLM via Groq API 
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format= "parsed")

chain = (
    {"context": retriver | format_docs, "question": RunnalvePassthrough()}
    | prompt
    | llm
    | StrOutputHandler()
)

print ("RAG chain assembled")



ImportError: cannot import name 'StrOutputHandler' from 'langchain_core.output_parsers' (/Users/nanuganti/mydev/ai-shopping-assistant/.venv/lib/python3.12/site-packages/langchain_core/output_parsers/__init__.py)